# Feature Engineering — Rossmann Sales Forecasting

## What is Feature Engineering?
Transforming raw data into meaningful input features
that help ML model understand patterns better.

## What we do in this notebook:
1. Load cleaned data from EDA
2. Extract date/time features
3. Create lag features (past sales)
4. Create rolling average features
5. Encode categorical variables
6. Save final feature set for model training

## Why this matters:
Raw data → Model = poor results
Raw data → Feature Engineering → Model = much better results

## Step 1 — Import Libraries

In [55]:
# ── Standard libraries ───────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Display settings ─────────────────────────────────────────────
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


## Step 2 — Load Cleaned Data
Loading the cleaned and merged dataset saved from 01_EDA.ipynb

In [56]:
# ── Load cleaned data from processed folder ───────────────────────
df = pd.read_csv('../data/processed/merged.csv')
# ── Convert Date to datetime ──────────────────────────────────────
df['Date'] = pd.to_datetime(df['Date'])

# # ── Filter only open stores ───────────────────────────────────────
# # Closed stores have Sales = 0, not useful for forecasting
# df = df[df['Open'] == 1].copy()

# ── Sort by Store and Date — critical for lag features ────────────
df = df.sort_values(['Store', 'Date']).reset_index(drop=True)

# ── Verify ────────────────────────────────────────────────────────
# print(f"📦 Shape after filtering closed stores : {df.shape}")
print(f"📅 Date range : {df['Date'].min()} → {df['Date'].max()}")
print(f"🏪 Total stores : {df['Store'].nunique()}")
print()
print("✅ Data loaded and ready for feature engineering!")

📅 Date range : 2013-01-01 00:00:00 → 2015-07-31 00:00:00
🏪 Total stores : 1115

✅ Data loaded and ready for feature engineering!


In [57]:
test = df[df['Date']>'2015-06-19'].copy()
print(test.columns)
test.to_csv('../data/processed/test.csv', index=False)

print(f"✅ Test data saved successfully!")

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval'],
      dtype='object')
✅ Test data saved successfully!


In [58]:
df = df[df['Date']<='2015-06-19'].copy()

## Step 3 — Date & Time Features
Extracting meaningful time-based features from the Date column.
These help the model understand seasonality and time patterns
we discovered during EDA.

In [59]:
# ── Extract date features ─────────────────────────────────────────
df['Year']        = df['Date'].dt.year
df['Month']       = df['Date'].dt.month
df['Day']         = df['Date'].dt.day
df['Week']        = df['Date'].dt.isocalendar().week.astype(int)
df['DayOfYear']   = df['Date'].dt.dayofyear
df['Quarter']     = df['Date'].dt.quarter

# ── Season feature ────────────────────────────────────────────────
# Based on EDA — seasons affect sales significantly
def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

df['Season'] = df['Month'].apply(get_season)

# ── Is month start/end feature ────────────────────────────────────
df['IsMonthStart'] = df['Date'].dt.is_month_start.astype(int)
df['IsMonthEnd']   = df['Date'].dt.is_month_end.astype(int)

# ── Is weekend feature ────────────────────────────────────────────
df['IsWeekend'] = (df['DayOfWeek'] >= 6).astype(int)

# ── Verify ────────────────────────────────────────────────────────
new_cols = ['Year', 'Month', 'Day', 'Week', 'DayOfYear',
            'Quarter', 'Season', 'IsMonthStart',
            'IsMonthEnd', 'IsWeekend']

print("📋 New date features created:")
print("-" * 35)
for col in new_cols:
    print(f"   ✅ {col:<20} — sample: {df[col].iloc[0]}")

print(f"\n📦 Shape now : {df.shape}")

📋 New date features created:
-----------------------------------
   ✅ Year                 — sample: 2013
   ✅ Month                — sample: 1
   ✅ Day                  — sample: 1
   ✅ Week                 — sample: 1
   ✅ DayOfYear            — sample: 1
   ✅ Quarter              — sample: 1
   ✅ Season               — sample: Winter
   ✅ IsMonthStart         — sample: 1
   ✅ IsMonthEnd           — sample: 0
   ✅ IsWeekend            — sample: 0

📦 Shape now : (970379, 28)


## Step 4 — Lag Features
Lag features = past sales values as input features.
These are the most powerful features for sales forecasting.

Example:
- Lag_1 = sales from previous day
- Lag_7  = sales from 7 days ago (same day last week)
- Lag_14 = sales from 14 days ago
- Lag_28 = sales from 28 days ago (same day 4 weeks ago)

Why they work:
→ Yesterday's sales predict today's sales
→ Same weekday last week is strong signal
→ Model learns "if last week was high, this week likely high too"

In [60]:
# ── Create lag features per store ─────────────────────────────────
# IMPORTANT: groupby Store — lags must be within same store!
# Otherwise Store 1's sales leak into Store 2's features

lag_days = [1, 7, 14, 21, 28]

for lag in lag_days:
    df[f'Lag_{lag}'] = df.groupby('Store')['Sales'].shift(lag)
    print(f"✅ Lag_{lag:<3} created")

print()
print(f"📦 Shape now : {df.shape}")
print()

# ── Preview lag features ──────────────────────────────────────────
print("📋 Sample — Store 1 first few rows with lags:")
sample = df[df['Store'] == 1][
    ['Date', 'Sales', 'Lag_1' ,'Lag_7', 'Lag_14', 'Lag_21', 'Lag_28']
].head(35).tail(5)
print(sample.to_string())

✅ Lag_1   created
✅ Lag_7   created
✅ Lag_14  created
✅ Lag_21  created
✅ Lag_28  created

📦 Shape now : (970379, 33)

📋 Sample — Store 1 first few rows with lags:
         Date  Sales   Lag_1   Lag_7  Lag_14  Lag_21  Lag_28
30 2013-01-31   4709 4601.00 5195.00 4044.00 4892.00 4327.00
31 2013-02-01   5633 4709.00 5586.00 4127.00 4881.00 4486.00
32 2013-02-02   5970 5633.00 5598.00 5182.00 4952.00 4997.00
33 2013-02-03      0 5970.00    0.00    0.00    0.00    0.00
34 2013-02-04   7032    0.00 4055.00 5394.00 4717.00 7176.00


## Step 5 — Rolling Average Features
Rolling averages smooth out daily fluctuations and capture
the recent sales trend for each store.

Example:
- Rolling_7  = average sales over last 7 days
- Rolling_14 = average sales over last 14 days
- Rolling_28 = average sales over last 28 days

Why they work:
→ Captures recent sales momentum
→ Smooths out one-off spikes or dips
→ Tells model "is this store trending up or down lately?"

In [61]:
# ── Rolling average features per store ───────────────────────────
# min_periods=1 means calculate even if fewer days available
windows = [7, 14, 28]

for window in windows:
    df[f'Rolling_Mean_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(
            window=window, min_periods=1).mean())
    )
    df[f'Rolling_Std_{window}'] = (
        df.groupby('Store')['Sales']
        .transform(lambda x: x.shift(1).rolling(
            window=window, min_periods=1).std())
    )
    print(f"✅ Rolling_Mean_{window} & Rolling_Std_{window} created")

print()
print(f"📦 Shape now : {df.shape}")
print()

# ── Preview rolling features ──────────────────────────────────────
print("📋 Sample — Store 1 rolling features:")
sample = df[df['Store'] == 1][
    ['Date', 'Sales', 'Rolling_Mean_7',
     'Rolling_Mean_14', 'Rolling_Mean_28']
].head(35).tail(5)
print(sample.to_string())

✅ Rolling_Mean_7 & Rolling_Std_7 created
✅ Rolling_Mean_14 & Rolling_Std_14 created
✅ Rolling_Mean_28 & Rolling_Std_28 created

📦 Shape now : (970379, 39)

📋 Sample — Store 1 rolling features:
         Date  Sales  Rolling_Mean_7  Rolling_Mean_14  Rolling_Mean_28
30 2013-01-31   4709         4108.57          4200.36          4221.14
31 2013-02-01   5633         4039.14          4247.86          4234.79
32 2013-02-02   5970         4045.86          4355.43          4275.75
33 2013-02-03      0         4099.00          4411.71          4310.50
34 2013-02-04   7032         4099.00          4411.71          4310.50


## Step 6 — Encode Categorical Variables
Categorical variables must be represented in a numerical form that is consistent between training and prediction. We use fixed mappings so that the same category always receives the same value across training, validation, and future forecasting.

Columns to encode:
- StoreType      : a, b, c, d → 0, 1, 2, 3
- Assortment     : a, b, c → 0, 1, 2
- StateHoliday   : 0, a, b, c → 0, 1, 2, 3
- Season         : Winter, Spring, Summer, Autumn → 0,1,2,3
- PromoInterval  : None, Jan,Apr... → numerical

In [62]:
print(df['StateHoliday'].unique())
print(df['StoreType'].unique())
print(df['Assortment'].unique())
print(df['PromoInterval'].unique())
print(df['Season'].unique())

['a' '0' 0 'b' 'c']
['c' 'a' 'd' 'b']
['a' 'c' 'b']
['None' 'Jan,Apr,Jul,Oct' 'Feb,May,Aug,Nov' 'Mar,Jun,Sept,Dec']
['Winter' 'Spring' 'Summer' 'Autumn']


In [63]:
CAT_MAPPINGS = {
    'Season': {'Winter': 0, 'Spring': 1, 'Summer': 2, 'Autumn': 3},
    'StoreType': {'a': 0, 'b': 1, 'c': 2, 'd': 3},
    'Assortment': {'a': 0, 'b': 1, 'c': 2},
    'StateHoliday': {'0': 0, 'a': 1, 'b': 2, 'c': 3},
    'PromoInterval': {
        'None': 0,
        'Jan,Apr,Jul,Oct': 1,
        'Feb,May,Aug,Nov': 2,
        'Mar,Jun,Sept,Dec': 3
    }
}

for col, mapping in CAT_MAPPINGS.items():
    df[col] = df[col].astype(str).map(mapping)

for col in df.columns:
    if df[col].dtype == 'object':
        print(f"⚠️ Column '{col}' is still object type!")

## Step 7 — Handle NaN from Lag Features & Final Cleanup
Lag features create NaN for first few rows of each store
(no past data available yet). We need to handle these
before saving the final dataset.

In [64]:
# ── Check NaN created by lag/rolling features ─────────────────────
print("📋 NaN count per lag/rolling feature:")
print("-" * 45)
lag_roll_cols = [c for c in df.columns if
                 'Lag_' in c or 'Rolling_' in c]
for col in lag_roll_cols:
    null_count = df[col].isnull().sum()
    print(f"   {col:<25} : {null_count:,} NaN")

print()

# ── Drop rows where lag features are NaN ─────────────────────────
# First 28 rows of each store will have NaN lags
df_clean = df.dropna(subset=lag_roll_cols).copy()

print(f"📦 Shape before dropna : {df.shape}")
print(f"📦 Shape after dropna  : {df_clean.shape}")
print(f"🗑️  Rows removed        : {df.shape[0] - df_clean.shape[0]:,}")
print()

# ── Final NaN check ───────────────────────────────────────────────
total_nan = df_clean.isnull().sum().sum()
print(f"✅ Total NaN remaining : {total_nan}")
print()
print("✅ Data is clean and ready to save!")

📋 NaN count per lag/rolling feature:
---------------------------------------------
   Lag_1                     : 1,115 NaN
   Lag_7                     : 7,805 NaN
   Lag_14                    : 15,610 NaN
   Lag_21                    : 23,415 NaN
   Lag_28                    : 31,220 NaN
   Rolling_Mean_7            : 1,115 NaN
   Rolling_Std_7             : 2,230 NaN
   Rolling_Mean_14           : 1,115 NaN
   Rolling_Std_14            : 2,230 NaN
   Rolling_Mean_28           : 1,115 NaN
   Rolling_Std_28            : 2,230 NaN



📦 Shape before dropna : (970379, 39)
📦 Shape after dropna  : (939159, 39)
🗑️  Rows removed        : 31,220

✅ Total NaN remaining : 0

✅ Data is clean and ready to save!


## Step 8 — Select Final Features & Save
Selecting the most relevant features for ML model
and saving the final dataset for model training.

In [65]:
# ── Define final feature columns ──────────────────────────────────
feature_cols = [
    # Target
    'Sales',

    # Store info
    'Store', 'Open', 'StoreType', 'Assortment',
    'CompetitionDistance', 'CompetitionOpenSinceMonth',
    'CompetitionOpenSinceYear',

    # Promo
    'Promo', 'Promo2', 'Promo2SinceWeek',
    'Promo2SinceYear', 'PromoInterval',

    # Date features
    'Year', 'Month', 'Day', 'Week',
    'DayOfWeek', 'DayOfYear', 'Quarter',
    'Season', 'IsWeekend', 'IsMonthStart', 'IsMonthEnd',

    # Holiday
    'StateHoliday', 'SchoolHoliday',

    # Lag features
    'Lag_1' ,'Lag_7', 'Lag_14', 'Lag_21', 'Lag_28',

    # Rolling features
    'Rolling_Mean_7', 'Rolling_Mean_14', 'Rolling_Mean_28',
    'Rolling_Std_7',  'Rolling_Std_14',  'Rolling_Std_28',
]

# ── Select final columns ──────────────────────────────────────────
df_final = df_clean[feature_cols].copy()

# ── Save to processed folder ──────────────────────────────────────
df_final.to_csv('../data/processed/train_features.csv', index=False)

# ── Verify ────────────────────────────────────────────────────────
import os
file_size = os.path.getsize(
    '../data/processed/train_features.csv') / (1024 * 1024)

print(f"📦 Final dataset shape  : {df_final.shape}")
print(f"📋 Total features       : {df_final.shape[1] - 1} + 1 target")
print(f"📁 File size            : {file_size:.2f} MB")
print()
print("📋 Feature Summary:")
print("-" * 40)
print(f"   Store features       : 6")
print(f"   Promo features       : 5")
print(f"   Date/Time features   : 10")
print(f"   Holiday features     : 2")
print(f"   Lag features         : 4")
print(f"   Rolling features     : 6")
print(f"   {'─'*25}")
print(f"   Total features       : 33")
print()
print("🎉 Feature Engineering COMPLETE!")

📦 Final dataset shape  : (939159, 37)
📋 Total features       : 36 + 1 target
📁 File size            : 196.13 MB

📋 Feature Summary:
----------------------------------------
   Store features       : 6
   Promo features       : 5
   Date/Time features   : 10
   Holiday features     : 2
   Lag features         : 4
   Rolling features     : 6
   ─────────────────────────
   Total features       : 33

🎉 Feature Engineering COMPLETE!
